# 05 - Evaluation
Load a trained NAFNet checkpoint, evaluate on the validation/test split, compute PSNR/SSIM (and optionally LPIPS), and save qualitative comparison figures to `outputs/`.

In [ ]:
import sys, os
sys.path.append('src')
import torch
from model import build_model
from preprocessing import train_val_split
from metrics import evaluate_batch, evaluate_lpips
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
data_root = "/content/dataset/train"  # adjust
checkpoint_path = "/content/drive/MyDrive/nafnet_best.pth"

model = build_model(in_channels=1, size='small').to(device)
ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

In [ ]:
_, val_ds = train_val_split(data_root, val_fraction=0.1)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

total_psnr, total_ssim, n = 0, 0, 0
with torch.no_grad():
    for noisy, clean in val_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        restored = model(noisy).clamp(0,1)
        m = evaluate_batch(restored, clean)
        total_psnr += m['psnr'] * noisy.size(0)
        total_ssim += m['ssim'] * noisy.size(0)
        n += noisy.size(0)

print(f"Val PSNR: {total_psnr/n:.2f} dB")
print(f"Val SSIM: {total_ssim/n:.4f}")

In [ ]:
# Qualitative comparison grid, saved to outputs/comparisons/
noisy, clean = next(iter(val_loader))
noisy, clean = noisy.to(device), clean.to(device)
with torch.no_grad():
    restored = model(noisy).clamp(0,1)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i in range(4):
    axes[0,i].imshow(noisy[i,0].cpu(), cmap='gray'); axes[0,i].set_title("Degraded"); axes[0,i].axis('off')
    axes[1,i].imshow(restored[i,0].cpu(), cmap='gray'); axes[1,i].set_title("Restored"); axes[1,i].axis('off')
    axes[2,i].imshow(clean[i,0].cpu(), cmap='gray'); axes[2,i].set_title("Clean GT"); axes[2,i].axis('off')
plt.tight_layout()
os.makedirs("outputs/comparisons", exist_ok=True)
plt.savefig("outputs/comparisons/comparison_grid.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save metrics summary
import json
summary = {"val_psnr_db": total_psnr/n, "val_ssim": total_ssim/n}
os.makedirs("outputs/metrics", exist_ok=True)
with open("outputs/metrics/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(summary)